In [1]:
import sys
sys.path.append("..")   # taaki src folder import ho sake

import ee, geemap
from src.config import GEE_PROJECT, AOI_BOX

ee.Initialize(project=GEE_PROJECT)
aoi = ee.Geometry.Rectangle(AOI_BOX)
print("Ready")

Ready


In [2]:
def s1_collection(start, end, orbit):
    return (ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.eq("orbitProperties_pass", orbit))
            .select("VV"))

def list_dates(start, end, orbit):
    col = s1_collection(start, end, orbit)
    dates = col.aggregate_array("system:time_start").map(
        lambda t: ee.Date(t).format("YYYY-MM-dd"))
    return sorted(set(dates.getInfo()))

for orbit in ["ASCENDING", "DESCENDING"]:
    print(orbit, list_dates("2022-08-15", "2022-10-15", orbit))

ASCENDING ['2022-08-18', '2022-08-30', '2022-09-11', '2022-09-23', '2022-10-05']
DESCENDING ['2022-08-16', '2022-08-21', '2022-08-28', '2022-09-02', '2022-09-14', '2022-09-21', '2022-09-26', '2022-10-03', '2022-10-08']


In [3]:
def flood_map(before, after, orbit, diff_thr=-3, water_thr=-15):
    b = s1_collection(*before, orbit).mean().focal_median(30, "circle", "meters")
    a = s1_collection(*after, orbit).mean().focal_median(30, "circle", "meters")

    diff = a.subtract(b)
    # naya paani = bahut dark ho gaya (diff) AND ab bhi absolute dark hai
    flood = diff.lt(diff_thr).And(a.lt(water_thr))

    # permanent paani (nadi, samudra) hatao
    perm = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").gt(50)
    flood = flood.updateMask(perm.Not())

    # steep slope hatao
    slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003"))
    flood = flood.updateMask(slope.lt(5)).selfMask()
    return b.clip(aoi), a.clip(aoi), flood.clip(aoi)

# NOTE: ye dates sirf example hain, Step 2 ke output se replace karo
BEFORE = ("2022-08-20", "2022-09-05")
AFTER  = ("2022-09-14", "2022-09-28")
ORBIT  = "DESCENDING"

before_img, after_img, flood = flood_map(BEFORE, AFTER, ORBIT)

area = (flood.multiply(ee.Image.pixelArea())
        .reduceRegion(ee.Reducer.sum(), aoi, 30, maxPixels=1e10)
        .get("VV"))
print("Flooded area (sq km):", ee.Number(area).divide(1e6).getInfo())

Flooded area (sq km): 17.245230027533797


In [4]:
m = geemap.Map(center=[20.55, 86.6], zoom=9)
m.addLayer(before_img, {"min": -25, "max": 0}, "Before")
m.addLayer(after_img, {"min": -25, "max": 0}, "After")
m.addLayer(flood, {"palette": ["blue"]}, "Flood")
m

Map(center=[20.55, 86.6], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [5]:
# Baseline 1: monsoon se pehle, dry season (kam geele khet)
BASELINES = {
    "dry (Mar-Apr)": ("2022-03-01", "2022-04-15"),
    "early monsoon (Aug 4-16)": ("2022-08-04", "2022-08-17"),
}

AFTER_WINDOWS = [
    ("2022-08-16", "2022-08-28"),
    ("2022-08-28", "2022-09-09"),
    ("2022-09-09", "2022-09-21"),
    ("2022-09-21", "2022-10-03"),
    ("2022-10-03", "2022-10-15"),
]

def flood_area(before, after, orbit="DESCENDING"):
    _, _, fl = flood_map(before, after, orbit)
    a = (fl.multiply(ee.Image.pixelArea())
         .reduceRegion(ee.Reducer.sum(), aoi, 30, maxPixels=1e10)
         .get("VV"))
    return round(ee.Number(a).divide(1e6).getInfo(), 1)

for bname, b in BASELINES.items():
    print("Baseline:", bname)
    for w in AFTER_WINDOWS:
        print("   ", w, "->", flood_area(b, w), "sq km")

Baseline: dry (Mar-Apr)
    ('2022-08-16', '2022-08-28') -> 407.3 sq km
    ('2022-08-28', '2022-09-09') -> 191.0 sq km
    ('2022-09-09', '2022-09-21') -> 18.6 sq km
    ('2022-09-21', '2022-10-03') -> 48.7 sq km
    ('2022-10-03', '2022-10-15') -> 38.7 sq km
Baseline: early monsoon (Aug 4-16)
    ('2022-08-16', '2022-08-28') -> 287.6 sq km
    ('2022-08-28', '2022-09-09') -> 112.4 sq km
    ('2022-09-09', '2022-09-21') -> 7.2 sq km
    ('2022-09-21', '2022-10-03') -> 18.0 sq km
    ('2022-10-03', '2022-10-15') -> 15.2 sq km


In [6]:
BEFORE = ("2022-08-04", "2022-08-17")
AFTER  = ("2022-08-18", "2022-08-24")

for orbit in ["ASCENDING", "DESCENDING"]:
    try:
        b, a, fl = flood_map(BEFORE, AFTER, orbit)
        area = (fl.multiply(ee.Image.pixelArea())
                .reduceRegion(ee.Reducer.sum(), aoi, 30, maxPixels=1e10)
                .get("VV"))
        print(orbit, "->", round(ee.Number(area).divide(1e6).getInfo(), 1), "sq km")
    except Exception as e:
        print(orbit, "error:", e)

ASCENDING -> 341.2 sq km
DESCENDING -> 29.9 sq km


In [7]:
ORBIT = "DESCENDING"   # ya "ASCENDING", jo zyada area de
b, a, fl = flood_map(BEFORE, AFTER, ORBIT)

m = geemap.Map(center=[20.55, 86.6], zoom=9)
m.addLayer(b, {"min": -25, "max": 0}, "Before")
m.addLayer(a, {"min": -25, "max": 0}, "After")
m.addLayer(fl, {"palette": ["blue"]}, "Flood")
m

Map(center=[20.55, 86.6], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [8]:
BEFORE = ("2022-08-04", "2022-08-17")
AFTER  = ("2022-08-18", "2022-08-24")
b, a, fl = flood_map(BEFORE, AFTER, "DESCENDING")

m = geemap.Map()
m.centerObject(aoi, 10)
m.addLayer(a, {"min": -25, "max": 0}, "After")
m.addLayer(fl, {"palette": ["0000FF"]}, "Flood")
m

Map(center=[20.550191737981482, 86.59999999999991], controls=(WidgetControl(options=['position', 'transparent_…

In [9]:
BEFORE = ("2022-08-04", "2022-08-17")
AFTER  = ("2022-08-18", "2022-08-24")

_, a_d, fl_d = flood_map(BEFORE, AFTER, "DESCENDING")
_, a_a, fl_a = flood_map(BEFORE, AFTER, "ASCENDING")

# dono ko jodo
flood_all = ee.ImageCollection([fl_d.unmask(0), fl_a.unmask(0)]).max().selfMask()

area = (flood_all.multiply(ee.Image.pixelArea())
        .reduceRegion(ee.Reducer.sum(), aoi, 30, maxPixels=1e10)
        .getInfo())
print(area)   # dictionary aayegi, uski value /1e6 karke sq km

m = geemap.Map()
m.centerObject(aoi, 9)
m.addLayer(a_d, {"min": -25, "max": 0}, "After DESC")
m.addLayer(a_a, {"min": -25, "max": 0}, "After ASC")
m.addLayer(flood_all, {"palette": ["0000FF"]}, "Flood (merged)")
m.addLayer(ee.Image().paint(aoi, 0, 2), {}, "AOI boundary")
m

{'VV': 351009629.62088937}


Map(center=[20.550191737981482, 86.59999999999991], controls=(WidgetControl(options=['position', 'transparent_…

In [10]:
# 8+ connected pixels (~0.7 hectare) se chhote patches hatao
conn = flood_all.connectedPixelCount(maxSize=100, eightConnected=True)
flood_clean = flood_all.updateMask(conn.gte(8))

def area_km2(img):
    d = (img.multiply(ee.Image.pixelArea())
         .reduceRegion(ee.Reducer.sum(), aoi, 30, maxPixels=1e10).getInfo())
    return round(list(d.values())[0] / 1e6, 1)

print("Merged flood :", area_km2(flood_all), "sq km")
print("Cleaned flood:", area_km2(flood_clean), "sq km")

m = geemap.Map()
m.centerObject(aoi, 10)
m.addLayer(a_d, {"min": -25, "max": 0}, "After DESC")
m.addLayer(flood_clean, {"palette": ["0000FF"]}, "Flood (clean)")
m

Merged flood : 351.0 sq km
Cleaned flood: 288.9 sq km


Map(center=[20.550191737981482, 86.59999999999991], controls=(WidgetControl(options=['position', 'transparent_…

In [11]:
task = ee.batch.Export.image.toDrive(
    image=flood_clean.unmask(0).toByte(),
    description="flood_mask_aug2022",
    folder="jalrakshak",
    fileNamePrefix="flood_mask_aug2022",
    region=aoi,
    scale=30,
    maxPixels=1e10,
)
task.start()
print("Export started. Check Drive folder 'jalrakshak' in 2-5 minutes.")

Export started. Check Drive folder 'jalrakshak' in 2-5 minutes.
